# Mathematical association between non-stationary evolution and clock violation

This notebook assembles a single multi-panel figure:

- **A** (row 1, left) — stationary process: $\mu^\prime$ (left y-axis) and ENS (right y-axis).
- **B** (row 1, right) — non-stationary process: $\mu^\prime$ (left y-axis) and ENS (right y-axis).
- **C** (row 2) — evolutionary-rate change under the three scenarios from
  `get_evolutionary_rate_change_plot`.

The figure is displayed and written to the manuscript's `1-math/math_fig.pdf`. Run with
`nbks/` as the working directory so `import plot_utils...` resolves.

In [1]:
import numpy as np
import plotly.graph_objects as go
from cogent3.maths.matrix_exponential_integration import expected_number_subs
from plot_utils.math import get_evolutionary_rate_change_plot
from plot_utils.project_paths import ROOT_DIR, write_pdf
from plot_utils.util import update_figure_format
from plotly.subplots import make_subplots

from clock_project.maths.evolutionary_rate import (
    calculate_non_stationary_rate,
    calculate_stationary_distribution,
    calculate_stationary_rate,
)

In [ ]:
MU_COLOR = "#0033FE"
ENS_COLOR = "#C60101"
X_TITLE = "<b><i>t</i></b>"
MU_TITLE = "\U0001d707′"  # mathematical-italic mu with a prime
ENS_TITLE = "\U0001d565"  # blackboard-bold t; MathJax lacks \mathbbm


def get_combined_rate_ens_plot(Q, t_range, pi=None):
    """Plot mu (left y-axis) and ENS (right y-axis) against time on one figure.

    When pi is None the process is treated as stationary, otherwise pi is the
    initial nucleotide frequency of a non-stationary process.
    """
    if pi is None:
        mu_values = [calculate_stationary_rate(Q) for _ in t_range]
        pi_ens = calculate_stationary_distribution(Q)
        ens_values = [-np.sum(pi_ens * np.diag(Q)) * t for t in t_range]
    else:
        mu_values = [calculate_non_stationary_rate(Q, pi, t) for t in t_range]
        ens_values = [expected_number_subs(pi, Q, t) for t in t_range]

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Scatter(
            x=t_range,
            y=mu_values,
            mode="lines",
            name=MU_TITLE,
            line={"color": MU_COLOR, "width": 4},
        ),
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=t_range,
            y=ens_values,
            mode="lines",
            name=ENS_TITLE,
            line={"color": ENS_COLOR, "width": 4},
        ),
        secondary_y=True,
    )

    fig.update_layout(title=None, title_font={"size": 20})
    fig.update_xaxes(title_text=X_TITLE)
    fig.update_yaxes(title_text=MU_TITLE, secondary_y=False)
    fig.update_yaxes(title_text=ENS_TITLE, secondary_y=True)
    fig = update_figure_format(fig)

    # colour each y-axis to match its trace and drop the right-axis grid
    fig.update_yaxes(
        title_font={"color": MU_COLOR},
        tickfont={"color": MU_COLOR},
        secondary_y=False,
    )
    fig.update_yaxes(
        title_font={"color": ENS_COLOR},
        tickfont={"color": ENS_COLOR},
        showgrid=False,
        secondary_y=True,
    )
    return fig

## Input data

In [3]:
t_range = np.linspace(0, 10, 999)
Q1 = (
    np.array(
        [
            [-0.14, 0.01, 0.04, 0.09],
            [0.4, -0.69, 0.09, 0.2],
            [0.63, 0.2, -1.3, 0.3],
            [0.07, 0.01, 0.02, -0.1],
        ]
    )
    * 0.3
)

Q2 = np.array(
    [
        [-0.242, 0.197, 0.008, 0.036],
        [0.17, -0.204, 0.004, 0.03],
        [0.044, 0.067, -0.137, 0.025],
        [0.018, 0.017, 0.146, -0.18],
    ]
)

Q3 = np.array(
    [
        [-0.633, 0.528, 0.006, 0.099],
        [0.111, -0.174, 0.017, 0.046],
        [0.133, 0.094, -0.312, 0.085],
        [0.028, 0.023, 0.045, -0.096],
    ]
)
pi = np.array([0.05, 0.35, 0.35, 0.25])

## Multi-panel figure

Row 1: stationary (**A**) and non-stationary (**B**) processes, each combining $\mu^\prime$ (left
y-axis) and ENS (right y-axis). Row 2 (**C**): the three evolutionary-rate change scenarios.

In [4]:
def get_math_figure():
    # Panels A and B: the two dual-axis combined plots.
    stationary_plot = get_combined_rate_ens_plot(Q1, t_range)
    non_stationary_plot = get_combined_rate_ens_plot(Q1, t_range, pi)

    # Panel C: the three evolutionary-rate change scenarios (mu in MU_COLOR).
    rate_change_plot = get_evolutionary_rate_change_plot(
        Q1, Q2, Q3, pi, t_range, color=MU_COLOR
    )

    # Explicit x-domains so the A/B gap is wide enough for A's right (ENS) axis
    # and B's left (mu) axis to sit side by side without overlapping. Row 2 keeps
    # its own three-column layout independent of that gap.
    a_dom, b_dom = [0.0, 0.40], [0.60, 1.0]
    c_doms = [[0.0, 0.28], [0.36, 0.64], [0.72, 1.0]]
    vertical_spacing = 0.176

    # 2 rows x 6 cols: row 1 holds A (cols 1-3) and B (cols 4-6), each with a
    # secondary y-axis; row 2 holds the three scenarios (cols 1-2, 3-4, 5-6).
    fig = make_subplots(
        rows=2,
        cols=6,
        specs=[
            [
                {"colspan": 3, "secondary_y": True},
                None,
                None,
                {"colspan": 3, "secondary_y": True},
                None,
                None,
            ],
            [
                {"colspan": 2},
                None,
                {"colspan": 2},
                None,
                {"colspan": 2},
                None,
            ],
        ],
        vertical_spacing=vertical_spacing,
    )

    # A and B: mu on the primary y-axis, ENS on the secondary y-axis.
    for col, panel, dom in (
        (1, stationary_plot, a_dom),
        (4, non_stationary_plot, b_dom),
    ):
        mu_trace, ens_trace = panel.data
        fig.add_trace(mu_trace, row=1, col=col, secondary_y=False)
        fig.add_trace(ens_trace, row=1, col=col, secondary_y=True)
        fig.update_xaxes(title_text=X_TITLE, domain=dom, row=1, col=col)
        fig.update_yaxes(title_text=MU_TITLE, row=1, col=col, secondary_y=False)
        fig.update_yaxes(
            title_text=ENS_TITLE, showgrid=False, row=1, col=col, secondary_y=True
        )

    # C: one mu scenario trace per column, mu labelled on the left-most only.
    for i, col in enumerate((1, 3, 5)):
        fig.add_trace(rate_change_plot.data[i], row=2, col=col)
        fig.update_xaxes(title_text=X_TITLE, domain=c_doms[i], row=2, col=col)
    fig.update_yaxes(title_text=MU_TITLE, row=2, col=1)

    fig = update_figure_format(fig)
    fig.update_layout(width=1200, height=850, showlegend=False)

    # Enlarge all axis titles (1.5x the update_figure_format default of 20),
    # leaving tick fonts unchanged.
    fig.update_xaxes(title_font_size=30)
    fig.update_yaxes(title_font_size=30)

    # Colour the axis titles/ticks to match their axes (mu/ENS); done last because
    # update_figure_format resets y-axis title colours to black.
    for col in (1, 4):
        fig.update_yaxes(
            title_font={"size": 30, "color": MU_COLOR},
            tickfont={"color": MU_COLOR},
            row=1,
            col=col,
            secondary_y=False,
        )
        fig.update_yaxes(
            title_font={"size": 30, "color": ENS_COLOR},
            tickfont={"color": ENS_COLOR},
            row=1,
            col=col,
            secondary_y=True,
        )

    # Row 2 mu axes: same colouring as the row-1 mu axis for consistency.
    for col in (1, 3, 5):
        fig.update_yaxes(tickfont={"color": MU_COLOR}, row=2, col=col)
    fig.update_yaxes(title_font={"size": 30, "color": MU_COLOR}, row=2, col=1)

    # Subplot titles (centred on each panel) and panel letters (top-left). Row-2
    # y-positions track the row-2 top so they stay attached as vertical_spacing changes.
    row2_top = (1 - vertical_spacing) / 2
    titles = [
        ("<b>Stationary</b>", sum(a_dom) / 2, 1.0),
        ("<b>Non-stationary</b>", sum(b_dom) / 2, 1.0),
        ("<b>Decrease</b>", sum(c_doms[0]) / 2, row2_top + 0.02),
        ("<b>Increase</b>", sum(c_doms[1]) / 2, row2_top + 0.02),
        ("<b>Non-monotonic</b>", sum(c_doms[2]) / 2, row2_top + 0.02),
    ]
    for text, x, y in titles:
        fig.add_annotation(
            text=text,
            xref="paper",
            yref="paper",
            x=x,
            y=y,
            xanchor="center",
            yanchor="bottom",
            showarrow=False,
            font={"size": 30, "family": "Times New Roman", "color": "black"},
        )

    for letter, (x, y) in zip(
        ("A", "B", "C"),
        ((a_dom[0], 1.03), (b_dom[0], 1.03), (c_doms[0][0], row2_top + 0.05)),
    ):
        fig.add_annotation(
            text=f"<b>{letter}</b>",
            xref="paper",
            yref="paper",
            x=x,
            y=y,
            xanchor="right",
            yanchor="bottom",
            showarrow=False,
            font={"size": 24, "family": "Times New Roman", "color": "black"},
        )
    return fig


math_figure = get_math_figure()
math_figure.show()

In [5]:
# Write the figure into the manuscript's figure tree using the pdf_writer helper.
math_fig_path = (
    ROOT_DIR.parent
    / "MolClock-MS"
    / "Figures"
    / "Result_figures"
    / "result_plots"
    / "1-math"
    / "math_fig.pdf"
)
write_pdf(math_figure, math_fig_path, width=1200, height=850)